In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from PIL import Image
from tqdm import tqdm

# --- 1. CONFIGURAZIONE ---
PATHS = {
    'train_img': '/kaggle/input/dataset/archive/images',
    'train_mask': '/kaggle/input/dataset/archive/masks',
    'labels': '/kaggle/input/dataset/archive/train_labels.csv',
    'test_img': '/kaggle/input/dataset/archive (1)/dataset challenge 2 dl /test_data/images',
    'test_mask': '/kaggle/input/dataset/archive (1)/dataset challenge 2 dl /test_data/masks'
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 12 # Bastano poche epoche per OvR
LR = 1e-4

# --- 2. PREPROCESSING ---
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])
    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except: return img_rgb

reinhard = ReinhardNormalizer()

def get_bbox_from_mask(mask):
    if np.sum(mask) == 0: return None
    rows, cols = np.where(mask > 0)
    return np.min(rows), np.max(rows), np.min(cols), np.max(cols)

class OvRTissueDataset(Dataset):
    def __init__(self, dataframe, img_dir, mask_dir, target_class_idx=None, transform=None, is_test=False):
        self.df = dataframe
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.is_test = is_test
        self.target_class_idx = target_class_idx

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['sample_index'] if not self.is_test else row['filename']
        mask_name = img_name.replace("img", "mask")
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        try:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        except:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            mask = None

        if mask is not None:
            bbox = get_bbox_from_mask(mask)
            if bbox:
                y_min, y_max, x_min, x_max = bbox
                image = image[y_min:y_max+1, x_min:x_max+1]
        
        image = reinhard(image)
        image = Image.fromarray(image)
        if self.transform: image = self.transform(image)
        
        if self.is_test: return image, img_name
        else:
            label = 1.0 if row['label_encoded'] == self.target_class_idx else 0.0
            return image, torch.tensor(label, dtype=torch.float32)

def get_transforms():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(90),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]), transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

# --- 3. TRAINING OVR ---
def train_ovr_classifier(target_class_idx, class_name, df):
    print(f"\n🚀 Training {class_name} (Class {target_class_idx})")
    
    # 1. Sampler Bilanciato (Cruciale)
    df['binary_target'] = (df['label_encoded'] == target_class_idx).astype(int)
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['binary_target'], random_state=42)
    
    weights = 1. / np.bincount(train_df['binary_target'])
    sample_weights = weights[train_df['binary_target'].values]
    sampler = WeightedRandomSampler(sample_weights, len(train_df))
    
    train_loader = DataLoader(OvRTissueDataset(train_df, PATHS['train_img'], PATHS['train_mask'], target_class_idx, get_transforms()[0]), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(OvRTissueDataset(val_df, PATHS['train_img'], PATHS['train_mask'], target_class_idx, get_transforms()[1]), 
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # 2. Modello Semplice
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 1)
    model = model.to(DEVICE)
    
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()
    
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        for imgs, lbls in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE).unsqueeze(1)
                preds = (torch.sigmoid(model(imgs)) > 0.5).float()
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(lbls.cpu().numpy())
        
        f1 = f1_score(val_targets, val_preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f"ovr_model_{target_class_idx}.pth")
            
    print(f"✅ Best F1: {best_f1:.4f}")

# --- 4. EXECUTION ---
def run():
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)
    
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df['label'])
    
    for idx, name in enumerate(le.classes_):
        train_ovr_classifier(idx, name, df)
    
    # --- SUBMISSION CON CALIBRAZIONE ---
    print("\nGenerating Submission...")
    test_files = [f for f in os.listdir(PATHS['test_img']) if f.endswith('.png')]
    test_df = pd.DataFrame({'filename': test_files})
    
    ds = OvRTissueDataset(test_df, PATHS['test_img'], PATHS['test_mask'], transform=get_transforms()[1], is_test=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    
    models_list = []
    for i in range(len(le.classes_)):
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, 1)
        m.load_state_dict(torch.load(f"ovr_model_{i}.pth"))
        m.to(DEVICE).eval()
        models_list.append(m)
        
    preds, fnames = [], []
    with torch.no_grad():
        for imgs, names in tqdm(loader):
            imgs = imgs.to(DEVICE)
            # Calcola le 4 probabilità
            probs = []
            for m in models_list:
                probs.append(torch.sigmoid(m(imgs)))
            
            # Stack [Batch, 4] e Argmax
            all_probs = torch.cat(probs, dim=1)
            preds.extend(le.inverse_transform(all_probs.argmax(1).cpu().numpy()))
            fnames.extend(names)
            
    pd.DataFrame({'sample_index': fnames, 'label': preds}).to_csv('submission.csv', index=False)
    print("Done.")

if __name__ == "__main__":
    run()


🚀 Training HER2(+) (Class 0)


✅ Best F1: 0.4194

🚀 Training Luminal A (Class 1)


✅ Best F1: 0.5098

🚀 Training Luminal B (Class 2)


✅ Best F1: 0.4962

🚀 Training Triple negative (Class 3)


✅ Best F1: 0.3913

Generating Submission...


100%|██████████| 15/15 [00:47<00:00,  3.15s/it]

Done.
